# My Code 

Main soft-label cutoff training experiment:
Controlled CTransPath experiment with corrected union-area soft labels, square-root BCE, no augmentation, Dynamic 1:1 sampler, seed 42, and F1-oriented checkpoint selection.

Training / validation loss uses continuous soft labels after cutoff filtering.
Validation / test metrics use original 50% hard labels for fair comparison.

Cutoff rule: soft values below the cutoff are set to 0; soft values equal to or above the cutoff keep their original continuous value.


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from timm.models.layers.helpers import to_2tuple
import timm
import random
import io
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
import os
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
from sklearn.metrics import precision_score, recall_score, f1_score

# =========================
# Reproducibility control
# =========================

SEED = 42

def set_seed(seed=42):
    import os
    import random
    import numpy as np
    import torch

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # More deterministic CUDA behaviour
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print(f"Controlled experiment seed fixed at: {SEED}")

# =========================
# Soft-label cutoff experiment config
# =========================

# Main cutoff variable.
# For the next experiments, change only this value to 0.1, 0.2, or 0.3,
# then restart kernel and run all cells.
SOFT_LABEL_CUTOFF = 0.3

# Cutoff rule:
# soft value < SOFT_LABEL_CUTOFF  -> 0
# soft value >= SOFT_LABEL_CUTOFF -> keep original continuous value
CUTOFF_TAG = f"cutoff{int(round(SOFT_LABEL_CUTOFF * 10)):02d}"
COVERTYPE = f"SoftCutoff{int(round(SOFT_LABEL_CUTOFF * 10)):02d}"

RUN_NAME = f"controlled_soft_{CUTOFF_TAG}_seed42_50_sqr_no_aug_aucselect"

# DICC project paths
PROJECT_ROOT = Path("/home/user/jiangjie/Jiangjie_Project")
HARD_LABEL_DIR = PROJECT_ROOT / "data" / "ResearchProject_50"
SOFT_LABEL_DIR = PROJECT_ROOT / "data" / "ResearchProject_soft_labels"
CHECKPOINT_PATH = PROJECT_ROOT / "weights" / "ctranspath.pth"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "soft_label_experiments" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kee baseline settings kept unchanged
NUM_CLASSES = 12
EPOCHS = 50
LEARNING_RATE = 1e-5
BATCH_SIZE = 128
PATIENCE = 3
NUM_WORKERS = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
print(f"RUN_NAME: {RUN_NAME}")
print(f"SOFT_LABEL_CUTOFF: {SOFT_LABEL_CUTOFF}")
print(f"CUTOFF_TAG: {CUTOFF_TAG}")
print(f"COVERTYPE: {COVERTYPE}")
print(f"HARD_LABEL_DIR exists: {HARD_LABEL_DIR.exists()} -> {HARD_LABEL_DIR}")
print(f"SOFT_LABEL_DIR exists: {SOFT_LABEL_DIR.exists()} -> {SOFT_LABEL_DIR}")
print(f"CHECKPOINT_PATH exists: {CHECKPOINT_PATH.exists()} -> {CHECKPOINT_PATH}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

# 12 OED morphological feature labels used in Kee CTransPath baseline
label_columns = [
    "Irregular epithelial stratification",
    "Loss of polarity of basal cells",
    "Drop shaped rete ridges",
    "Premature keratinization in single cells",
    "Loss of epithelial cell cohesion",
    "Abnormal variation in nuclear size",
    "Abnormal variation in nuclear shape",
    "Abnormal variation in cell size",
    "Abnormal variation in cell shape",
    "Increased N:C ratio",
    "Increased number and size of nucleoli",
    "Hyperchromasia",
]

print("Number of labels:", len(label_columns))

class PathBlur(object):
    def __init__(self, blur_sigma=(0.5, 2.0), poisson_scale=(5, 20), jpeg_quality=(30, 90)):
        self.blur_sigma = blur_sigma
        self.poisson_scale = poisson_scale
        self.jpeg_quality = jpeg_quality

    def __call__(self, img):
        # Gaussian blur
        sigma = random.uniform(*self.blur_sigma)
        img = img.filter(ImageFilter.GaussianBlur(radius=sigma))

        # Poisson noise
        np_img = np.array(img).astype(np.float32)
        scale = random.uniform(*self.poisson_scale)
        noisy = np.random.poisson(np_img * scale) / scale
        noisy = np.clip(noisy, 0, 255).astype(np.uint8)
        img = Image.fromarray(noisy)

        # JPEG compression artifact
        buffer = io.BytesIO()
        quality = random.randint(*self.jpeg_quality)
        img.save(buffer, format='JPEG', quality=quality)
        img = Image.open(buffer)

        return img

# Fixed pre-processing applied to all data
base_transforms = [
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
]

# Validation/Test Transform: no augmentation
val_test_transform = transforms.Compose(base_transforms)

# Keep the no-augmentation baseline setting unchanged
AUGMENTATION_COMBINATIONS = {
    'None': val_test_transform,
}


/home/user/jiangjie/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Controlled experiment seed fixed at: 42
Using device: cuda
RUN_NAME: controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect
SOFT_LABEL_CUTOFF: 0.3
CUTOFF_TAG: cutoff03
COVERTYPE: SoftCutoff03
HARD_LABEL_DIR exists: True -> /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50
SOFT_LABEL_DIR exists: True -> /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels
CHECKPOINT_PATH exists: True -> /home/user/jiangjie/Jiangjie_Project/weights/ctranspath.pth
OUTPUT_DIR: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect
Number of labels: 12


In [2]:
# =========================
# Transform setting: Run A = no augmentation
# =========================

base_transforms = [
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
]

# Validation/Test Transform: no augmentation
val_test_transform = transforms.Compose(base_transforms)

# Run A: 50% Square-root Weight, no augmentation
AUGMENTATION_COMBINATIONS = {
    "None": val_test_transform
}

print("Augmentation setting:")
print(AUGMENTATION_COMBINATIONS)

Augmentation setting:
{'None': Compose(
    Resize(size=224, interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
)}


In [3]:
# @title Original code

# --- 1. CTransPath Architecture (Swin + ConvStem) ---

class ConvStem(nn.Module):

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=768, norm_layer=None, flatten=True):
        super().__init__()

        assert patch_size == 4
        assert embed_dim % 8 == 0

        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.flatten = flatten


        stem = []
        input_dim, output_dim = 3, embed_dim // 8
        for l in range(2):
            stem.append(nn.Conv2d(input_dim, output_dim, kernel_size=3, stride=2, padding=1, bias=False))
            stem.append(nn.BatchNorm2d(output_dim))
            stem.append(nn.ReLU(inplace=True))
            input_dim = output_dim
            output_dim *= 2
        stem.append(nn.Conv2d(input_dim, embed_dim, kernel_size=1))
        self.proj = nn.Sequential(*stem)

        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x):
        B, C, H, W = x.shape
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x)
        if self.flatten:
            x = x.flatten(2).transpose(1, 2)  # BCHW -> BNC
        x = self.norm(x)
        return x

def ctranspath(num_classes, checkpoint_path=CHECKPOINT_PATH):
    # Load the Swin-Tiny model structure and inject the ConvStem
    model = timm.create_model(
        model_name="swin_tiny_patch4_window7_224",
        embed_layer=ConvStem, # Your custom class defined elsewhere
        pretrained=False,       # Instructs timm to download and load the weights
        num_classes=0, # Remove default head
    )

    # --- MANUAL CHECKPOINT LOADING ---
    if checkpoint_path:
        print(f"Loading weights manually from: {checkpoint_path}")
        # Assuming the checkpoint is a dict with the model state under the key 'model'
        state_dict = torch.load(checkpoint_path, map_location='cpu')

        # We need to filter the state_dict because the head layer dimensions won't match
        # CTransPath checkpoints usually save the entire model
        # We try to load the full state dict and ignore the mismatched head
        model.load_state_dict(state_dict, strict=False)

    # Ensure Global Average Pooling is explicitly enabled for feature extraction
    model.global_pool = 'avg'
    in_features = model.num_features

    # Replace the classification head for 12 OED classes
    model.head = nn.Linear(in_features, num_classes)

    return model

    # # --- MANUAL CHECKPOINT LOADING ---
    # if checkpoint_path:
    #     print(f"Loading weights manually from: {checkpoint_path}")
    #     # Assuming the checkpoint is a dict with the model state under the key 'model'
    #     state_dict = torch.load(checkpoint_path, map_location='cpu')

    #     # We need to filter the state_dict because the head layer dimensions won't match
    #     # CTransPath checkpoints usually save the entire model
    #     model.load_state_dict(state_dict, strict=False)

    # # Replace the classification head for 12 OED classes
    # # This is a standard fine-tuning step.
    # in_features = model.head.in_features
    # model.head = nn.Linear(in_features, num_classes)
    # return model

# def ctranspath(num_classes, checkpoint_path=CHECKPOINT_PATH):
#     # 1. Create the model structure
#     model = timm.create_model(
#         model_name="swin_tiny_patch4_window7_224",
#         embed_layer=ConvStem,
#         pretrained=False,
#         num_classes=0, # Remove default head
#     )

#     # 2. Load Checkpoint (Backbone Weights)
#     if checkpoint_path:
#         print(f"Loading weights manually from: {checkpoint_path}")
#         state_dict = torch.load(checkpoint_path, map_location='cpu')
#         # Filter state_dict to match non-strict loading if needed
#         model.load_state_dict(state_dict['model'], strict=False)

#     # 3. FREEZE THE BACKBONE (Crucial for Small Data)
#     print("❄️ Freezing ALL CTransPath backbone layers...")
#     for param in model.parameters():
#         param.requires_grad = False

#     # # 4. UNFREEZE THE LAST BLOCK (The "Fine-Tuning" Step)
#     # # This allows the model to learn specific histology textures
#     # print("🔓 Unfreezing the last Swin Transformer block...")

#     # # Unfreeze the last layer block (layers.3 in Swin Tiny)
#     # for param in model.layers[-1].parameters():
#     #     param.requires_grad = True

#     # # Unfreeze the final normalization layer
#     # for param in model.norm.parameters():
#     #     param.requires_grad = True

#     # 5. Add the Head (Trainable)
#     model.global_pool = 'avg'
#     in_features = model.num_features

#     model.head = nn.Sequential(
#         nn.Linear(in_features, 256),
#         nn.ReLU(),
#         nn.Dropout(p=0.5),
#         nn.Linear(256, num_classes)
#     )
#     # Ensure head is trainable
#     for param in model.head.parameters():
#         param.requires_grad = True

#     return model

# --- Helper Functions ---
def to_2tuple(x):
    if isinstance(x, (tuple, list)):
        return tuple(x)
    return (x, x)

def collate_fn(batch):
    # Filters out samples where image loading failed (returned None)
    batch = [item for item in batch if item[0] is not None]
    if not batch: return None, None
    return torch.utils.data.dataloader.default_collate(batch)


In [4]:
# =========================
# Model output shape check
# =========================

model = ctranspath(NUM_CLASSES, checkpoint_path=CHECKPOINT_PATH).to(device)
model.eval()

dummy = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    out = model(dummy)

print("Model output shape:", out.shape)
assert out.shape == (2, NUM_CLASSES), f"Unexpected output shape: {out.shape}"

print("Model output shape check passed.")

/home/user/jiangjie/.local/lib/python3.10/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading weights manually from: /home/user/jiangjie/Jiangjie_Project/weights/ctranspath.pth
Model output shape: torch.Size([2, 12])
Model output shape check passed.


In [5]:
# --- 3. Custom Dataset ---

class MultiLabelTileDataset(Dataset):
    def __init__(self, df, label_columns, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_columns = label_columns
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Image path used directly from the DataFrame (assuming full path)
        img_path = self.df.loc[idx, 'filepath']

        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"File not found: {img_path}")
            return None, None

        labels = self.df.loc[idx, self.label_columns].values.astype('float32')
        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(labels, dtype=torch.float32)

        return image, labels

In [6]:
# =========================
# Load soft-label train/validation data and hard-label evaluation data
# =========================

weight_type = "Sqr"
covertype = COVERTYPE

# Training uses the ten soft-label train split files explicitly
# These ten files are the source files that were merged into final_df_train_soft_overlap.csv.
soft_train_files = [
    SOFT_LABEL_DIR / f"final_df_train{i}_soft_overlap.csv"
    for i in range(1, 11)
]

# Validation loss uses continuous union-area soft labels
soft_val_file = SOFT_LABEL_DIR / "final_df_val_soft_overlap.csv"

# Validation/test metrics still use original Kee 50% hard labels
hard_val_file = HARD_LABEL_DIR / "final_df_val.csv"
hard_test_file = HARD_LABEL_DIR / "final_df_test.csv"

print("Checking CSV files:")
for f in soft_train_files + [soft_val_file, hard_val_file, hard_test_file]:
    print(f.name, "exists:", f.exists(), "->", f)

def fix_filepath(df):
    df = df.copy()

    replacements = [
        ("/content/ResearchProject_50", str(HARD_LABEL_DIR)),
        ("/content/ResearchProject_soft_labels", str(SOFT_LABEL_DIR)),
        ("/scr/user/jiangjie/Jiangjie_Project", str(PROJECT_ROOT)),
    ]

    df["filepath"] = df["filepath"].astype(str)

    for old_prefix, new_prefix in replacements:
        df["filepath"] = df["filepath"].str.replace(
            old_prefix,
            new_prefix,
            regex=False
        )

    return df

# Load the ten soft-label train split CSVs and concatenate them
train_parts = []

for f in soft_train_files:
    temp_df = pd.read_csv(f, low_memory=False)
    temp_df = fix_filepath(temp_df)
    train_parts.append(temp_df)

train_df = pd.concat(train_parts, ignore_index=True)

# Load validation soft labels for validation loss
val_soft_df = fix_filepath(pd.read_csv(soft_val_file, low_memory=False))

# Load original 50% hard labels for validation/test metrics
val_hard_df = fix_filepath(pd.read_csv(hard_val_file, low_memory=False))
test_hard_df = fix_filepath(pd.read_csv(hard_test_file, low_memory=False))

# Compatibility aliases for later checking cells
val_df = val_hard_df
test_df = test_hard_df

print("\nLoaded data:")
print("train_df soft:", train_df.shape)
print("val_soft_df:", val_soft_df.shape)
print("val_hard_df:", val_hard_df.shape)
print("test_hard_df:", test_hard_df.shape)

# Confirm the ten soft train files were concatenated correctly
expected_train_rows = 135946 * 10
assert len(train_df) == expected_train_rows, f"Unexpected train_df rows: {len(train_df)}"

# Soft labels must remain float32 continuous values in [0, 1]
for df in [train_df, val_soft_df]:
    for col in label_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(np.float32)

def apply_soft_label_cutoff(df, label_columns, cutoff, name):
    """
    Apply soft-label cutoff filtering.

    Rule:
    soft value < cutoff  -> 0
    soft value >= cutoff -> keep original continuous value

    This is not binarization. Values above the cutoff remain continuous.
    """
    df = df.copy()
    values = df[label_columns].values.astype(np.float32)

    before_nonzero = int((values > 0).sum())
    before_mass = float(values.sum())

    values[values < cutoff] = 0.0
    values = np.clip(values, 0.0, 1.0).astype(np.float32)

    df[label_columns] = values

    after_nonzero = int((values > 0).sum())
    after_mass = float(values.sum())

    print("\n" + "=" * 80)
    print(f"Soft-label cutoff applied to {name}")
    print("Cutoff:", cutoff)
    print("Rule: soft < cutoff -> 0; soft >= cutoff -> keep original value")
    print("Nonzero label values before:", before_nonzero)
    print("Nonzero label values after :", after_nonzero)
    print("Removed nonzero values     :", before_nonzero - after_nonzero)
    print("Soft positive mass before :", round(before_mass, 4))
    print("Soft positive mass after  :", round(after_mass, 4))
    print("Mass removed              :", round(before_mass - after_mass, 4))

    return df

# Apply cutoff to the soft labels used by the loss function.
# train_df: training loss target.
# val_soft_df: validation loss target, kept consistent with the training target distribution.
# Hard validation/test labels remain original 50% binary labels for metrics.
train_df = apply_soft_label_cutoff(train_df, label_columns, SOFT_LABEL_CUTOFF, "train_df")
val_soft_df = apply_soft_label_cutoff(val_soft_df, label_columns, SOFT_LABEL_CUTOFF, "val_soft_df")

# Hard labels must remain original 50% binary labels for fair evaluation
for df in [val_hard_df, test_hard_df]:
    for col in label_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

print("\nExample filepath after replacement:")
print(train_df["filepath"].iloc[0])
print("Exists:", Path(train_df["filepath"].iloc[0]).exists())

assert Path(train_df["filepath"].iloc[0]).exists(), "Filepath replacement failed."

# Sanity check: label distribution
def check_label_distribution(df, name, label_columns):
    label_values = df[label_columns].values

    print("\n" + "=" * 80)
    print(name)
    print("Shape:", df.shape)
    print("Label dtype:", label_values.dtype)
    print("Min:", np.nanmin(label_values))
    print("Max:", np.nanmax(label_values))
    print("Has NaN:", np.isnan(label_values.astype(float)).any())

    sample_values = label_values[:min(200000, len(label_values))]
    unique_sample = np.unique(sample_values)
    print("Number of unique values in sample:", len(unique_sample))
    print("First 20 unique values:", unique_sample[:20])

    non_binary_count = int(np.sum((label_values > 0) & (label_values < 1)))
    print("Number of non-binary soft label values:", non_binary_count)

check_label_distribution(train_df, "Soft Train Labels", label_columns)
check_label_distribution(val_soft_df, "Soft Validation Labels", label_columns)
check_label_distribution(val_hard_df, "Hard Validation Labels", label_columns)
check_label_distribution(test_hard_df, "Hard Test Labels", label_columns)

# WSI-level split check
train_slides = set(train_df["slide_name"].dropna().unique())
val_slides = set(val_hard_df["slide_name"].dropna().unique())
test_slides = set(test_hard_df["slide_name"].dropna().unique())

print("\nUnique WSIs:")
print("train:", len(train_slides))
print("val:", len(val_slides))
print("test:", len(test_slides))
print("all:", len(train_slides | val_slides | test_slides))

print("\nOverlap check:")
print("train ∩ val :", sorted(train_slides & val_slides))
print("train ∩ test:", sorted(train_slides & test_slides))
print("val ∩ test  :", sorted(val_slides & test_slides))


Checking CSV files:
final_df_train1_soft_overlap.csv exists: True -> /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train1_soft_overlap.csv
final_df_train2_soft_overlap.csv exists: True -> /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train2_soft_overlap.csv
final_df_train3_soft_overlap.csv exists: True -> /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train3_soft_overlap.csv
final_df_train4_soft_overlap.csv exists: True -> /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train4_soft_overlap.csv
final_df_train5_soft_overlap.csv exists: True -> /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train5_soft_overlap.csv
final_df_train6_soft_overlap.csv exists: True -> /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_train6_soft_overlap.csv
final_df_train7_soft_overlap.csv exists: True -> /home/user/jiangjie

In [7]:
# =========================
# Dynamic 1:1 sampler + square-root pos_weight using cutoff-filtered soft-label training data
# =========================

# Define abnormal patch for soft-label training:
# after cutoff filtering, any tile-label pair with soft value > 0 is treated as abnormal for the sampler.
train_df["is_abnormal_12"] = (train_df[label_columns].sum(axis=1) > 0).astype(int)

n_abnormal = int(train_df["is_abnormal_12"].sum())
n_normal = int(len(train_df) - n_abnormal)

print("Training patch distribution based on cutoff-filtered soft labels:")
print("Total train patches:", len(train_df))
print("Normal train patches:", n_normal)
print("Abnormal train patches:", n_abnormal)
print("Abnormal percentage:", round(n_abnormal / len(train_df) * 100, 4), "%")

assert n_abnormal > 0, "No abnormal soft-label tiles found. Please check soft-label CSV."
assert n_normal > 0, "No normal tiles found. Please check soft-label CSV."

# Dynamic 1:1 sampling weights
sample_weights = np.zeros(len(train_df), dtype=np.float32)

sample_weights[train_df["is_abnormal_12"].values == 1] = 1.0 / n_abnormal
sample_weights[train_df["is_abnormal_12"].values == 0] = 1.0 / n_normal

# Each epoch contains all abnormal patches and an equal number of sampled normal patches
num_train_samples = int(n_abnormal * 2)

sampler_generator = torch.Generator()
sampler_generator.manual_seed(SEED)

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=num_train_samples,
    replacement=True,
    generator=sampler_generator
)

print("\nDynamic 1:1 sampler:")
print("num_train_samples per epoch:", num_train_samples)
print("Expected abnormal samples per epoch:", n_abnormal)
print("Expected normal samples per epoch:", n_abnormal)

# Square-root pos_weight calculation based on soft-label positive mass
positive_counts = train_df[label_columns].sum(axis=0).astype(float)

# Effective negatives under balanced epoch setting
effective_total = num_train_samples
effective_negatives = effective_total - positive_counts

raw_pos_weight = effective_negatives / (positive_counts + 1e-6)
sqrt_pos_weight = np.sqrt(raw_pos_weight)

pos_weight_tensor = torch.tensor(
    sqrt_pos_weight.values,
    dtype=torch.float32
).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

weight_table = pd.DataFrame({
    "OED_feature": label_columns,
    "soft_positive_mass": positive_counts.values,
    "effective_total_per_epoch": effective_total,
    "effective_negatives": effective_negatives.values,
    "raw_pos_weight": raw_pos_weight.values,
    "sqrt_pos_weight": sqrt_pos_weight.values,
})

print("\nSquare-root pos_weight table:")
display(weight_table)

weight_table_path = OUTPUT_DIR / f"soft_{CUTOFF_TAG}_sqrt_pos_weight_table.csv"
weight_table.to_csv(weight_table_path, index=False)
print("Saved pos_weight table:", weight_table_path)


Training patch distribution based on cutoff-filtered soft labels:
Total train patches: 1359460
Normal train patches: 1332720
Abnormal train patches: 26740
Abnormal percentage: 1.967 %

Dynamic 1:1 sampler:
num_train_samples per epoch: 53480
Expected abnormal samples per epoch: 26740
Expected normal samples per epoch: 26740

Square-root pos_weight table:


,OED_feature,soft_positive_mass,effective_total_per_epoch,effective_negatives,raw_pos_weight,sqrt_pos_weight
0,Irregular epithelial stratification,429.803192,53480,53050.196808,123.429043,11.109862
1,Loss of polarity of basal cells,1967.447144,53480,51512.552856,26.182433,5.116877
2,Drop shaped rete ridges,705.339844,53480,52774.660156,74.821606,8.649948
3,Premature keratinization in single cells,7352.683105,53480,46127.316895,6.273535,2.504703
4,Loss of epithelial cell cohesion,5379.306152,53480,48100.693848,8.941803,2.990285
5,Abnormal variation in nuclear size,5243.055664,53480,48236.944336,9.200159,3.033176
6,Abnormal variation in nuclear shape,7763.236816,53480,45716.763184,5.888879,2.426701
7,Abnormal variation in cell size,311.980621,53480,53168.019379,170.420903,13.054536
8,Abnormal variation in cell shape,8431.174805,53480,45048.825195,5.343126,2.311520
9,Increased N:C ratio,3189.288086,53480,50290.711914,15.768633,3.970974


Saved pos_weight table: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/soft_cutoff03_sqrt_pos_weight_table.csv


In [8]:
# =========================
# Create Dataset and DataLoader + real batch check
# =========================

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

# Current augmentation setting: only one item, "None"
aug_name, train_transform = list(AUGMENTATION_COMBINATIONS.items())[0]

print("Current augmentation:", aug_name)
assert aug_name == "None", "Soft-Cutoff run should keep the no-augmentation baseline."

train_dataset = MultiLabelTileDataset(
    train_df,
    label_columns=label_columns,
    transform=train_transform
)

# Validation loss uses soft labels
val_soft_dataset = MultiLabelTileDataset(
    val_soft_df,
    label_columns=label_columns,
    transform=val_test_transform
)

# Validation metrics use original 50% hard labels
val_hard_dataset = MultiLabelTileDataset(
    val_hard_df,
    label_columns=label_columns,
    transform=val_test_transform
)

# Test metrics use original 50% hard labels
test_hard_dataset = MultiLabelTileDataset(
    test_hard_df,
    label_columns=label_columns,
    transform=val_test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

val_soft_loader = DataLoader(
    val_soft_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

val_hard_loader = DataLoader(
    val_hard_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

test_hard_loader = DataLoader(
    test_hard_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

print("Dataset sizes:")
print("train_dataset:", len(train_dataset))
print("val_soft_dataset:", len(val_soft_dataset))
print("val_hard_dataset:", len(val_hard_dataset))
print("test_hard_dataset:", len(test_hard_dataset))

print("\nDataLoader batches:")
print("train_loader batches per epoch:", len(train_loader))
print("val_soft_loader batches:", len(val_soft_loader))
print("val_hard_loader batches:", len(val_hard_loader))
print("test_hard_loader batches:", len(test_hard_loader))

# Check one real training batch
images, labels = next(iter(train_loader))

print("\nOne training batch:")
print("images shape:", images.shape)
print("labels shape:", labels.shape)
print("labels dtype:", labels.dtype)
print("labels min:", labels.min().item())
print("labels max:", labels.max().item())
print("non-binary labels in this batch:", int(((labels > 0) & (labels < 1)).sum().item()))

assert labels.dtype == torch.float32
assert labels.min().item() >= 0.0
assert labels.max().item() <= 1.0


Current augmentation: None
Dataset sizes:
train_dataset: 1359460
val_soft_dataset: 198083
val_hard_dataset: 198083
test_hard_dataset: 314284

DataLoader batches:
train_loader batches per epoch: 418
val_soft_loader batches: 1548
val_hard_loader batches: 1548
test_hard_loader batches: 2456

One training batch:
images shape: torch.Size([128, 3, 224, 224])
labels shape: torch.Size([128, 12])
labels dtype: torch.float32
labels min: 0.0
labels max: 1.0
non-binary labels in this batch: 33


## Checkpoint selection update

For the soft-label cutoff experiment, the primary checkpoint and early-stopping metric is validation hard-label **Macro AUC**. This is because the goal is to improve hard-label classification performance under severe class imbalance, not only ranking ability. Micro F1, Macro Recall, and Macro AUROC checkpoints are still saved as secondary references.

The cutoff is applied only to soft training labels and soft validation labels used for loss. Validation/test metrics still use the original 50% hard labels.


In [9]:
# @title Training and evaluation loop for soft-label training
# --- 4. Main Training and Evaluation Loop ---

def safe_macro_roc_auc(y_true, y_score):
    """
    Compute macro AUROC safely for multilabel data.
    Labels with only one class in y_true are skipped.
    """
    aucs = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[:, i], y_score[:, i]))

    if len(aucs) == 0:
        return np.nan

    return float(np.mean(aucs))

def safe_micro_roc_auc(y_true, y_score):
    """
    Compute micro AUROC safely.
    """
    try:
        return float(roc_auc_score(y_true, y_score, average="micro"))
    except ValueError:
        return np.nan

def collect_predictions(model, data_loader, device):
    model.eval()

    all_true = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in data_loader:
            if inputs is None:
                continue

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            probs = torch.sigmoid(outputs)

            all_true.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    all_true = np.concatenate(all_true, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)

    return all_true, all_probs

def evaluate_hard_label_metrics(y_true, y_probs, label_columns, threshold=0.5):
    """
    Evaluation metrics against original 50% hard labels.
    """
    y_true = y_true.astype(int)
    y_pred = (y_probs >= threshold).astype(int)

    micro_auc = safe_micro_roc_auc(y_true, y_probs)
    macro_auc = safe_macro_roc_auc(y_true, y_probs)

    micro_precision = precision_score(y_true, y_pred, average="micro", zero_division=0)
    micro_recall = recall_score(y_true, y_pred, average="micro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)

    macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    weighted_precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    weighted_recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    per_class_rows = []
    for i, label in enumerate(label_columns):
        if len(np.unique(y_true[:, i])) < 2:
            auc = np.nan
        else:
            auc = roc_auc_score(y_true[:, i], y_probs[:, i])

        per_class_rows.append({
            "Feature": label,
            "AUROC": auc,
            "Precision": precision_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "Recall": recall_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "F1": f1_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "Support": int(y_true[:, i].sum()),
            "Predicted Positive": int(y_pred[:, i].sum()),
        })

    summary = {
        "threshold": threshold,
        "micro_auroc": micro_auc,
        "macro_auroc": macro_auc,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
    }

    per_class_df = pd.DataFrame(per_class_rows)

    return summary, per_class_df, y_pred

def train_and_evaluate_run(
    model_name,
    train_loader,
    val_loss_loader,
    val_metric_loader,
    test_loader,
    num_classes,
    criterion,
    optimizer_class,
    lr,
    num_epochs,
    patience,
    label_columns,
    checkpoint_path,
    weight_type,
    covertype
):

    print(f"\n--- Running Strategy: {model_name} ---")
    print("Training loss: cutoff-filtered soft labels")
    print("Validation loss: cutoff-filtered soft labels")
    print("Validation/test metrics: original 50% hard labels")
    print("Primary checkpoint selection: Val Hard Macro AUC")
    print("Secondary checkpoints saved for reference: Val Hard Micro F1@0.5, Val Hard Macro Recall@0.5, Val Hard Macro F1@0.5")

    # Initialize model for a fresh run
    model = ctranspath(num_classes, checkpoint_path=checkpoint_path).to(device)
    optimizer = optimizer_class(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    # Main model selection is based on validation hard-label Macro AUC.
    # Reason: this follows the original controlled CTransPath baseline protocol.
    # F1, recall, and threshold-based metrics are still monitored and saved as
    # secondary checkpoints, but they do not control early stopping.
    best_val_macro_auc = -1.0
    best_val_macro_f1 = -1.0
    best_val_micro_f1 = -1.0
    best_val_macro_recall = -1.0

    epochs_without_improvement = 0

    best_model_path = str(OUTPUT_DIR / f"best_model_{weight_type}_{covertype}_{model_name}_by_macro_auc.pth")
    best_macro_f1_model_path = str(OUTPUT_DIR / f"best_model_{weight_type}_{covertype}_{model_name}_by_macro_f1.pth")
    best_micro_f1_model_path = str(OUTPUT_DIR / f"best_model_{weight_type}_{covertype}_{model_name}_by_micro_f1.pth")
    best_macro_recall_model_path = str(OUTPUT_DIR / f"best_model_{weight_type}_{covertype}_{model_name}_by_macro_recall.pth")

    history_rows = []

    for epoch in range(num_epochs):
        # Training phase with soft labels
        model.train()
        running_loss = 0.0

        train_seen = 0

        for inputs, labels in tqdm(train_loader, desc=f"E {epoch+1}/{num_epochs} (Train Soft)"):
            if inputs is None:
                continue

            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            batch_size_actual = inputs.size(0)
            running_loss += loss.item() * batch_size_actual
            train_seen += batch_size_actual

        train_loss = running_loss / max(train_seen, 1)

        # Validation loss phase with soft labels
        model.eval()
        val_loss = 0.0

        val_seen = 0

        with torch.no_grad():
            for inputs, labels in tqdm(val_loss_loader, desc=f"E {epoch+1}/{num_epochs} (Val Soft Loss)"):
                if inputs is None:
                    continue

                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                batch_size_actual = inputs.size(0)
                val_loss += loss.item() * batch_size_actual
                val_seen += batch_size_actual

        val_loss /= max(val_seen, 1)

        # Validation metrics phase with hard labels
        val_true_hard, val_probs = collect_predictions(model, val_metric_loader, device)
        val_summary, _, _ = evaluate_hard_label_metrics(
            y_true=val_true_hard,
            y_probs=val_probs,
            label_columns=label_columns,
            threshold=0.5
        )

        val_micro_auc = val_summary["micro_auroc"]
        val_macro_auc = val_summary["macro_auroc"]
        val_micro_f1 = val_summary["micro_f1"]
        val_macro_f1 = val_summary["macro_f1"]
        val_micro_recall = val_summary["micro_recall"]
        val_macro_recall = val_summary["macro_recall"]

        print(
            f"| Epoch {epoch+1:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Soft Loss: {val_loss:.4f} | "
            f"Val Hard Micro AUC: {val_micro_auc:.4f} | "
            f"Val Hard Macro AUC: {val_macro_auc:.4f} | "
            f"Val Hard Micro F1@0.5: {val_summary['micro_f1']:.4f} | "
            f"Val Hard Macro F1@0.5: {val_summary['macro_f1']:.4f} |"
        )

        history_rows.append({
            "epoch": epoch + 1,
            "train_loss_soft": train_loss,
            "val_loss_soft": val_loss,
            **{f"val_{k}": v for k, v in val_summary.items()}
        })

        # ---------------------------------------------------------
        # Checkpoint selection
        # ---------------------------------------------------------
        # Primary early-stopping metric: validation hard-label Macro AUC.
        # Secondary checkpoints are saved for reference only.

        current_macro_auc = val_macro_auc
        current_macro_f1 = val_macro_f1
        current_micro_f1 = val_micro_f1
        current_macro_recall = val_macro_recall

        if np.isnan(current_macro_auc):
            current_macro_auc = -1.0
        if np.isnan(current_macro_f1):
            current_macro_f1 = -1.0
        if np.isnan(current_micro_f1):
            current_micro_f1 = -1.0
        if np.isnan(current_macro_recall):
            current_macro_recall = -1.0

        # Secondary: save best Micro F1 model for comparison
        if current_micro_f1 > best_val_micro_f1:
            best_val_micro_f1 = current_micro_f1
            torch.save(model.state_dict(), best_micro_f1_model_path)
            print(f"Saved best Micro F1 model to: {best_micro_f1_model_path}")

        # Secondary: save best Macro Recall model for recall-oriented analysis
        if current_macro_recall > best_val_macro_recall:
            best_val_macro_recall = current_macro_recall
            torch.save(model.state_dict(), best_macro_recall_model_path)
            print(f"Saved best Macro Recall model to: {best_macro_recall_model_path}")

        # Secondary: save best Macro F1 model for threshold-based comparison
        if current_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = current_macro_f1
            torch.save(model.state_dict(), best_macro_f1_model_path)
            print(f"Saved best Macro F1 model to: {best_macro_f1_model_path}")

        # Primary: save best Macro AUC model and control early stopping
        if current_macro_auc > best_val_macro_auc:
            best_val_macro_auc = current_macro_auc
            torch.save(model.state_dict(), best_model_path)
            epochs_without_improvement = 0
            print(f"Saved best Macro AUC model to: {best_model_path}")
        else:
            epochs_without_improvement += 1
            print(f"No Macro AUC improvement. epochs_without_improvement = {epochs_without_improvement}")

            if epochs_without_improvement >= patience:
                print(f"Early stopping triggered. Best Macro AUC model saved at: {best_model_path}")
                break

    history_df = pd.DataFrame(history_rows)
    history_path = OUTPUT_DIR / f"training_history_{weight_type}_{covertype}_{model_name}.csv"
    history_df.to_csv(history_path, index=False)
    print("Saved training history:", history_path)

    # Final evaluation on hard-label test set
    # Main reported test result uses the checkpoint selected by validation hard-label Macro AUC.
    print("Loading best Macro AUC checkpoint for final test evaluation:", best_model_path)
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()

    test_true_hard, test_probs = collect_predictions(model, test_loader, device)

    test_summary, test_per_class_df, test_preds_binary = evaluate_hard_label_metrics(
        y_true=test_true_hard,
        y_probs=test_probs,
        label_columns=label_columns,
        threshold=0.5
    )

    print("\n=== Per-label AUROC and Binary Metrics (Hard Test Set, threshold=0.5) ===")
    display(test_per_class_df)

    # Classification report
    print(f"\n--- Overall Classification Report for {model_name} (Hard Test Set) ---")
    report = classification_report(
        y_true=test_true_hard.astype(int),
        y_pred=test_preds_binary,
        target_names=label_columns,
        zero_division=0,
        output_dict=False
    )
    print(report)

    print("\n--- Final Test Results for", model_name, "---")
    for k, v in test_summary.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")
    print("-------------------------------------------------\n")

    # Save final outputs
    test_summary_df = pd.DataFrame([test_summary])
    test_summary_path = OUTPUT_DIR / f"test_summary_{weight_type}_{covertype}_{model_name}.csv"
    test_per_class_path = OUTPUT_DIR / f"test_per_class_{weight_type}_{covertype}_{model_name}.csv"
    test_probs_path = OUTPUT_DIR / f"test_probs_{weight_type}_{covertype}_{model_name}.npy"
    test_targets_path = OUTPUT_DIR / f"test_targets_hard_{weight_type}_{covertype}_{model_name}.npy"

    test_summary_df.to_csv(test_summary_path, index=False)
    test_per_class_df.to_csv(test_per_class_path, index=False)
    np.save(test_probs_path, test_probs)
    np.save(test_targets_path, test_true_hard)

    print("Saved test summary:", test_summary_path)
    print("Saved per-class metrics:", test_per_class_path)
    print("Saved test probabilities:", test_probs_path)
    print("Saved hard test targets:", test_targets_path)

    return test_summary, test_per_class_df


# Result of AUC 
Test Micro AUROC
Test Macro AUROC

In [10]:
# @title Soft-Cutoff Training Execution
# --- 5. Execution Block ---

set_seed(SEED)

weight_type = "Sqr"
covertype = COVERTYPE

final_results = {}

# Run loop kept for compatibility, but only "None" augmentation is enabled.
for name, train_transform in AUGMENTATION_COMBINATIONS.items():
    print("\n" + "=" * 100)
    print("Starting experiment:", RUN_NAME)
    print("Augmentation:", name)
    print("Soft-label cutoff:", SOFT_LABEL_CUTOFF)
    print("Cutoff mode: soft < cutoff -> 0; soft >= cutoff -> keep original value")
    print("Soft train CSVs:", [p.name for p in soft_train_files])
    print("Soft validation CSV for loss:", soft_val_file)
    print("Hard validation CSV for metrics:", hard_val_file)
    print("Hard test CSV for metrics:", hard_test_file)
    print("=" * 100)

    # Rebuild train loader for the current transform
    current_train_dataset = MultiLabelTileDataset(
        train_df,
        label_columns=label_columns,
        transform=train_transform
    )

    # Rebuild a fresh sampler for the formal training run.
    # The earlier DataLoader sanity check consumes the sampler generator state,
    # so rebuilding here keeps the official run reproducible from SEED.
    current_sampler_generator = torch.Generator()
    current_sampler_generator.manual_seed(SEED)

    current_sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=num_train_samples,
        replacement=True,
        generator=current_sampler_generator
    )

    current_loader_generator = torch.Generator()
    current_loader_generator.manual_seed(SEED)

    current_train_loader = DataLoader(
        current_train_dataset,
        batch_size=BATCH_SIZE,
        sampler=current_sampler,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True,
        shuffle=False,
        worker_init_fn=seed_worker,
        generator=current_loader_generator
    )

    test_summary, test_per_class_df = train_and_evaluate_run(
        model_name=name,
        train_loader=current_train_loader,
        val_loss_loader=val_soft_loader,
        val_metric_loader=val_hard_loader,
        test_loader=test_hard_loader,
        num_classes=NUM_CLASSES,
        criterion=criterion,
        optimizer_class=torch.optim.AdamW,
        lr=LEARNING_RATE,
        num_epochs=EPOCHS,
        patience=PATIENCE,
        label_columns=label_columns,
        checkpoint_path=CHECKPOINT_PATH,
        weight_type=weight_type,
        covertype=covertype
    )

    final_results[name] = test_summary

# Final compact summary
final_results_df = pd.DataFrame([
    {"Strategy": name, **metrics}
    for name, metrics in final_results.items()
])

final_results_path = OUTPUT_DIR / "final_results_summary.csv"
final_results_df.to_csv(final_results_path, index=False)

print("\n\n================== FINAL TEST SUMMARY ==================")
display(final_results_df)
print("Saved final results:", final_results_path)
print("========================================================")



Starting experiment: controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect
Augmentation: None
Soft-label cutoff: 0.3
Cutoff mode: soft < cutoff -> 0; soft >= cutoff -> keep original value
Soft train CSVs: ['final_df_train1_soft_overlap.csv', 'final_df_train2_soft_overlap.csv', 'final_df_train3_soft_overlap.csv', 'final_df_train4_soft_overlap.csv', 'final_df_train5_soft_overlap.csv', 'final_df_train6_soft_overlap.csv', 'final_df_train7_soft_overlap.csv', 'final_df_train8_soft_overlap.csv', 'final_df_train9_soft_overlap.csv', 'final_df_train10_soft_overlap.csv']
Soft validation CSV for loss: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_soft_labels/final_df_val_soft_overlap.csv
Hard validation CSV for metrics: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_val.csv
Hard test CSV for metrics: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_test.csv

--- Running Strategy: None ---
Training loss: cutoff-filtered soft labels
Validati

E 1/50 (Val Soft Loss): 100%|██████████| 1548/1548 [02:53<00:00,  8.91it/s]


| Epoch 01 | Train Loss: 0.4513 | Val Soft Loss: 0.2159 | Val Hard Micro AUC: 0.8521 | Val Hard Macro AUC: 0.8521 | Val Hard Micro F1@0.5: 0.0487 | Val Hard Macro F1@0.5: 0.0319 |
Saved best Micro F1 model to: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff03_None_by_micro_f1.pth
Saved best Macro Recall model to: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff03_None_by_macro_recall.pth
Saved best Macro F1 model to: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff03_None_by_macro_f1.pth
Saved best Macro AUC model to: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff03_None_by_macro_auc.pt

E 2/50 (Val Soft Loss): 100%|██████████| 1548/1548 [02:53<00:00,  8.93it/s]


| Epoch 02 | Train Loss: 0.4009 | Val Soft Loss: 0.2511 | Val Hard Micro AUC: 0.8582 | Val Hard Macro AUC: 0.8374 | Val Hard Micro F1@0.5: 0.0453 | Val Hard Macro F1@0.5: 0.0388 |
Saved best Macro Recall model to: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff03_None_by_macro_recall.pth
Saved best Macro F1 model to: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff03_None_by_macro_f1.pth
No Macro AUC improvement. epochs_without_improvement = 1


...

| Epoch 12 | Train Loss: 0.2910 | Val Soft Loss: 0.1448 | Val Hard Micro AUC: 0.8909 | Val Hard Macro AUC: 0.8817 | Val Hard Micro F1@0.5: 0.0826 | Val Hard Macro F1@0.5: 0.0820 |
No Macro AUC improvement. epochs_without_improvement = 3
Early stopping triggered. Best Macro AUC model saved at: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff03_None_by_macro_auc.pth
Saved training history: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/training_history_Sqr_SoftCutoff03_None.csv
Loading best Macro AUC checkpoint for final test evaluation: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/best_model_Sqr_SoftCutoff03_None_by_macro_auc.pth

=== Per-label AUROC and Binary Metrics (Hard Test Set, threshold=0.5) ===


,Feature,AUROC,Precision,Recall,F1,Support,Predicted Positive
0,Irregular epithelial stratification,0.864910,0.000000,0.000000,0.000000,53,250
1,Loss of polarity of basal cells,0.832251,0.002328,0.009434,0.003735,212,859
2,Drop shaped rete ridges,0.871562,0.003266,0.133333,0.006376,45,1837
3,Premature keratinization in single cells,0.944039,0.153332,0.343813,0.212081,3467,7774
4,Loss of epithelial cell cohesion,0.900179,0.041760,0.125844,0.062710,1629,4909
5,Abnormal variation in nuclear size,0.811256,0.000369,0.020690,0.000726,145,8124
6,Abnormal variation in nuclear shape,0.919649,0.043886,0.177669,0.070385,1424,5765
7,Abnormal variation in cell size,0.909971,0.000000,0.000000,0.000000,20,169
8,Abnormal variation in cell shape,0.883784,0.049459,0.167671,0.076387,1992,6753
9,Increased N:C ratio,0.661944,0.000612,0.028947,0.001198,380,17983



--- Overall Classification Report for None (Hard Test Set) ---
                                          precision    recall  f1-score   support

     Irregular epithelial stratification       0.00      0.00      0.00        53
         Loss of polarity of basal cells       0.00      0.01      0.00       212
                 Drop shaped rete ridges       0.00      0.13      0.01        45
Premature keratinization in single cells       0.15      0.34      0.21      3467
        Loss of epithelial cell cohesion       0.04      0.13      0.06      1629
      Abnormal variation in nuclear size       0.00      0.02      0.00       145
     Abnormal variation in nuclear shape       0.04      0.18      0.07      1424
         Abnormal variation in cell size       0.00      0.00      0.00        20
        Abnormal variation in cell shape       0.05      0.17      0.08      1992
                     Increased N:C ratio       0.00      0.03      0.00       380
   Increased number and size of n

,Strategy,threshold,micro_auroc,macro_auroc,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,None,0.5,0.909523,0.863919,0.03904,0.252543,0.067625,0.032259,0.130185,0.048522,0.081498,0.252543,0.12093


Saved final results: /home/user/jiangjie/Jiangjie_Project/outputs/soft_label_experiments/controlled_soft_cutoff03_seed42_50_sqr_no_aug_aucselect/final_results_summary.csv


In [11]:
print("--- Soft Positive Mass in CUTOFF-FILTERED TRAIN Set ---")
print(train_df[label_columns].sum())

print("\n--- Soft Positive Mass in CUTOFF-FILTERED SOFT VALIDATION Set ---")
print(val_soft_df[label_columns].sum())

print("\n--- Positive Class Counts in HARD VALIDATION Set ---")
print(val_hard_df[label_columns].sum())

print("\n--- Positive Class Counts in HARD TEST Set ---")
print(test_hard_df[label_columns].sum())


--- Soft Positive Mass in CUTOFF-FILTERED TRAIN Set ---
Irregular epithelial stratification          429.803192
Loss of polarity of basal cells             1967.447144
Drop shaped rete ridges                      705.339844
Premature keratinization in single cells    7352.683105
Loss of epithelial cell cohesion            5379.306152
Abnormal variation in nuclear size          5243.055664
Abnormal variation in nuclear shape         7763.236816
Abnormal variation in cell size              311.980621
Abnormal variation in cell shape            8431.174805
Increased N:C ratio                         3189.288086
Increased number and size of nucleoli       3931.254395
Hyperchromasia                              6257.783203
dtype: float32

--- Soft Positive Mass in CUTOFF-FILTERED SOFT VALIDATION Set ---
Irregular epithelial stratification          352.989075
Loss of polarity of basal cells              391.270966
Drop shaped rete ridges                       84.224854
Premature keratinizati

In [12]:
# --- Optional compact summary after training ---
print("\n\n================== FINAL TEST SUMMARY ==================")

if "final_results" in globals() and len(final_results) > 0:
    final_results_df = pd.DataFrame([
        {"Strategy": name, **metrics}
        for name, metrics in final_results.items()
    ])
    display(final_results_df)
else:
    print("final_results is not available yet. Please run the training execution cell first.")

print("========================================================")




================== FINAL TEST SUMMARY ==================


,Strategy,threshold,micro_auroc,macro_auroc,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,None,0.5,0.909523,0.863919,0.03904,0.252543,0.067625,0.032259,0.130185,0.048522,0.081498,0.252543,0.12093
